# Train giọng nói riêng bằng Piper TTS (fine-tune từ vais1000)

**Cách dùng:** chạy lần lượt từng ô từ trên xuống (bấm nút Play bên trái mỗi ô, hoặc menu Runtime > Run all).

Trước khi chạy: vào **Runtime (Thời gian chạy) > Change runtime type > GPU (T4)** để bật GPU miễn phí.

**Chuẩn bị trước:** nén thư mục `dataset` (chứa `wavs/` và `metadata.csv` do tool VoiceRecorder tạo ra) thành file `dataset.zip`, upload lên Google Drive, ghi nhớ đường dẫn (ví dụ: `MyDrive/dataset.zip`).

## 1. Kiểm tra GPU

In [ ]:
!nvidia-smi


## 2. Kết nối Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 3. Giải nén dataset từ Drive

Sửa đường dẫn `DATASET_ZIP` bên dưới cho đúng vị trí file `dataset.zip` bạn đã upload.

In [ ]:
DATASET_ZIP = "/content/drive/MyDrive/dataset.zip"  # sua duong dan neu can

import os, zipfile
os.makedirs("/content/dataset_raw", exist_ok=True)
with zipfile.ZipFile(DATASET_ZIP, 'r') as z:
    z.extractall("/content/dataset_raw")

# tim thu muc chua wavs/ va metadata.csv (phong truong hop zip co thu muc con)
import glob
candidates = glob.glob("/content/dataset_raw/**/metadata.csv", recursive=True)
assert len(candidates) > 0, "Khong tim thay metadata.csv trong file zip!"
DATASET_DIR = os.path.dirname(candidates[0])
print("Dataset dir:", DATASET_DIR)
print("So dong metadata:", sum(1 for _ in open(os.path.join(DATASET_DIR, "metadata.csv"), encoding="utf-8")))


## 4. Cài đặt Piper và các thư viện cần thiết

In [ ]:
!apt-get -qq install espeak-ng > /dev/null
!git clone -q https://github.com/rhasspy/piper.git /content/piper
!pip install -q -e /content/piper/src/python
!pip install -q huggingface_hub

# build phien ban monotonic align (can cho training)
%cd /content/piper/src/python
!bash build_monotonic_align.sh
%cd /content


## 5. Tải checkpoint tiếng Việt có sẵn (vais1000, medium) để fine-tune

In [ ]:
from huggingface_hub import list_repo_files, hf_hub_download

files = list_repo_files("rhasspy/piper-checkpoints", repo_type="dataset")
ckpt_candidates = [f for f in files if f.startswith("vi/vi_VN/vais1000/medium/") and f.endswith(".ckpt")]
print("Tim thay checkpoint:", ckpt_candidates)
assert len(ckpt_candidates) > 0, "Khong tim thay file .ckpt, kiem tra lai ten repo/duong dan."

ckpt_local = hf_hub_download("rhasspy/piper-checkpoints", ckpt_candidates[0],
                              repo_type="dataset", local_dir="/content/base_checkpoint")
print("Da tai ve:", ckpt_local)


## 6. Tiền xử lý dataset (preprocess)

Bước này convert audio + text sang định dạng Piper cần để train.

In [ ]:
!python3 -m piper_train.preprocess \
  --language vi \
  --input-dir "{DATASET_DIR}" \
  --output-dir /content/training_dir \
  --dataset-format ljspeech \
  --single-speaker \
  --sample-rate 22050


## 7. Fine-tune (train tiếp từ checkpoint có sẵn)

`--max_epochs` là **tổng số epoch cuối cùng** (không phải số epoch train thêm). Checkpoint vais1000 gốc thường đã ở epoch cao, nên đặt `max_epochs` cao hơn epoch hiện tại của checkpoint một khoảng ~1000.

Nếu Colab bị ngắt kết nối giữa chừng, chạy lại ô này với `--resume_from_checkpoint` trỏ tới checkpoint mới nhất trong `/content/training_dir/lightning_logs/version_0/checkpoints/` để train tiếp (không phải chạy lại từ đầu).

In [ ]:
!python3 -m piper_train \
  --dataset-dir /content/training_dir \
  --accelerator gpu \
  --devices 1 \
  --batch-size 16 \
  --validation-split 0.0 \
  --num-test-examples 0 \
  --max_epochs 3000 \
  --resume_from_checkpoint "{ckpt_local}" \
  --checkpoint-epochs 1 \
  --precision 32


## 7b. (Tuỳ chọn) Sao lưu checkpoint sang Drive định kỳ

Vì Colab free có thể ngắt phiên bất cứ lúc nào, nên chạy ô này SAU KHI dừng training (Runtime > Interrupt) để lưu tiến độ, tránh mất công train lại từ đầu.

In [ ]:
import shutil, os
os.makedirs("/content/drive/MyDrive/piper_backup", exist_ok=True)
!cp -r /content/training_dir/lightning_logs /content/drive/MyDrive/piper_backup/
print("Da sao luu checkpoint sang Google Drive: MyDrive/piper_backup/lightning_logs")


## 8. Export ra file .onnx

Tự động lấy checkpoint mới nhất (epoch cao nhất) để export.

In [ ]:
import glob, os

ckpts = glob.glob("/content/training_dir/lightning_logs/version_0/checkpoints/*.ckpt")
assert len(ckpts) > 0, "Chua co checkpoint nao duoc luu, kiem tra lai buoc training."
latest_ckpt = max(ckpts, key=os.path.getmtime)
print("Dung checkpoint:", latest_ckpt)

OUTPUT_NAME = "giong_toi"  # doi ten tuy y
!python3 -m piper_train.export_onnx "{latest_ckpt}" "/content/{OUTPUT_NAME}.onnx"

# copy file config.json ra thanh <ten>.onnx.json (Piper can file nay di kem)
!cp /content/training_dir/config.json "/content/{OUTPUT_NAME}.onnx.json"
print("Xong! File model: /content/{OUTPUT_NAME}.onnx va /content/{OUTPUT_NAME}.onnx.json")


## 9. Nghe thử kết quả

In [ ]:
TEST_TEXT = "Xin chào, đây là giọng nói do tôi tự huấn luyện."

!echo "{TEST_TEXT}" | python3 -m piper --model "/content/{OUTPUT_NAME}.onnx" --output_file /content/test_output.wav

from IPython.display import Audio
Audio("/content/test_output.wav")


## 10. Lưu kết quả cuối cùng vào Google Drive

Copy 2 file `.onnx` và `.onnx.json` này về Drive, sau đó tải về máy để dùng trong app .NET 9.

In [ ]:
import shutil
shutil.copy(f"/content/{OUTPUT_NAME}.onnx", f"/content/drive/MyDrive/{OUTPUT_NAME}.onnx")
shutil.copy(f"/content/{OUTPUT_NAME}.onnx.json", f"/content/drive/MyDrive/{OUTPUT_NAME}.onnx.json")
print("Da luu vao Google Drive: MyDrive/", OUTPUT_NAME + ".onnx", "va", OUTPUT_NAME + ".onnx.json")
